### Vision 모델 내부 동작원리
- 이미지 인코딩 : 이미지 임베딩 벡터화
- 프로젝션 : 이미지 벡터를 LLM 토큰 공간으로 변환
- 토큰 결합 : 시각 토큰 + 텍스트 토큰을 순서대로 배열
- LLM 처리
- 텍스트 생성

In [2]:
import base64

from langchain_ibm import ChatWatsonx, WatsonxEmbeddings
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_chroma import Chroma
import os
from langchain_ollama import ChatOllama
from pathlib import Path
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage

import requests
from io import BytesIO
from PIL import Image

c:\source\ollama\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
def encode_image(image_path:str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [6]:
vision_llm = ChatOllama(model="minimax-m3:cloud", temperature=0)

parser = StrOutputParser()

# 이미지 + 텍스트 : 이미지 기반으로 질문
def ask_about_image(image_path, question):
    image_b64 = encode_image(image_path)
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "human", [
                    {"type" : "image_url", "image_url" : {'url' : f'data:image/jpeg;base64,{image_b64}'}},
                    {"type" : "text", "text": question}
                ]
            )
        ]
    )

    chain = prompt | vision_llm | parser
    response = chain.invoke({"image_b64":image_b64, "question":question})

    return response

In [5]:
ask_about_image('./image/animals1.jpg', 'Which animal is in the image?')

"The image shows a **cat** — specifically a tabby cat with a classic striped coat pattern in shades of brown, black, and tan. Notable features include:\n\n- **Bright blue eyes** (a striking color for a tabby)\n- **Classic tabby markings** on the forehead\n- **Long white whiskers**\n- A slightly open mouth, as if it's meowing\n\nThe cat is photographed from a close-up, upward angle against a warm orange background."

In [7]:
vision_llm = ChatOllama(model="minimax-m3:cloud", temperature=0)

# 이미지 다운로드
url = "https://images.unsplash.com/photo-1770009079291-82b8a594ee23?q=80&w=709&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D"
img = requests.get(url).content

# 인코딩
img_b64 = base64.b64encode(img).decode()

# 모델
message = HumanMessage(
    content=[
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"},
        },
        {"type": "text", "text": "이 사진에 대해 자세히 설명과 분석을 해라"},
    ]
)

vision_llm.invoke([message]).content

"# 바다 위의 일출/일몰 사진 분석\n\n## 📸 장면 묘사\n\n이 사진은 **바다 위로 떠오르거나 지고 있는 태양**을 담고 있는 풍경 사진입니다. 태양이 화면 중앙 상단에 위치하며, 두꺼운 구름에 부분적으로 가려져 있습니다. 수평선 너머로 산이나 섬의 실루엣이 어둡게 드러나 있고, 화면의 하반부는 잔잔한 바다로 채워져 있습니다.\n\n## 🎨 구도 분석\n\n**삼분할 구도 (Rule of Thirds)**\n- 수평선이 화면 중앙 약간 아래에 위치하여 하늘과 바다가 균형 있게 배치됨\n- 태양이 상단 삼분할 지점 근처에 위치하여 시각적 중심 역할\n\n**수평선과 대칭**\n- 수평선을 기준으로 위(하늘)와 아래(바다)가 거의 대칭을 이루며, 바다 표면의 잔잔한 물결이 하늘의 구름을 반영하는 듯한 효과를 만들어냄\n\n## 🌅 빛과 색상 분석\n\n- **색온도**: 따뜻한 톤(웜톤)이 지배적 — 황금색, 호박색, 주황색 계열\n- **골든아워(Golden Hour)**: 일출 직후 또는 일몰 직전의 부드럽고 따뜻한 빛\n- **대비**: 밝은 태양 주변부와 어두운 구름, 그리고 물의 반영이 만들어내는 명도 대비\n- **빛의 확산**: 구름 가장자리로 퍼져나가는 빛이 '크레푸스큘러 선(Crepuscular rays, 신의 광선)' 같은 효과를 만들어 감성적인 분위기를 자아냄\n\n## ☁️ 구름과 하늘\n\n- 화면 우측에 큰 적운(적란운) 형태의 구름이 질량감을 가지고 자리 잡고 있음\n- 왼쪽과 위쪽에는 가늘고 길게 늘어진 권운(卷雲)이나 고적운이 분포\n- 구름 사이로 새어나오는 빛이 드라마틱한 분위기를 연출\n\n## 🌊 바다의 표현\n\n- 잔잔한 물결이 일렁이는 표면\n- 황금빛 하늘을 반사하며 따뜻한 색조를 띠고 있음\n- 강한 바람이 없는 평화로운 해상 상태\n\n## 🏔️ 원경의 실루엣\n\n- 오른쪽 수평선 너머로 산맥 또는 큰 섬의 윤곽이 검게 드러남\n- 이 요소가 평면적일 수 있는 구도에 깊이감과 입체감을 부여\n- 일본, 제

In [9]:
# 모든 이미지를 jpeg 저장 / 인코딩

def pil_to_base64(img:Image.Image, format="JPEG") -> str:
    buffer = BytesIO()
    img.save(buffer, format=format)

    return base64.b64encode(buffer.getvalue()).decode('utf-8')

In [10]:
img = Image.open("./image/system.png")

# 리사이즈
img_resized = img.resize((800,600))

# Gray
img_gray = img.convert('L')
img_b64 = pil_to_base64(img_gray)

message = HumanMessage(
    content=[
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"},
        },
        {"type": "text", "text": "이 사진에 대해 자세히 설명과 분석을 해라"},
    ]
)

result = vision_llm.invoke([message]).content
print(result)



# 채용공고 상세 분석: 미래로봇추진단 - 시스템 소프트웨어 개발

## 📋 공고 개요

이 이미지는 **미래로봇추진단(서울 군포)**에서 **S/W개발 - 시스템 소프트웨어** 포지션의 채용공고입니다. 로봇 산업 분야, 특히 휴머노이드 로봇의 핵심 소프트웨어를 개발하는 직무를 다루고 있습니다.

---

## 🔍 섹션별 상세 분석

### 1️⃣ 포지션 소개 (Job Overview)
> **"휴머노이드 로봇의 실시간 제어 시스템 및 소프트웨어 플랫폼을 개발하는 직무"**

- **핵심 키워드**: 운영체제 환경 구성, 디바이스 드라이버, 실시간 제어 프레임워크
- 로봇 동작의 **핵심 기반 소프트웨어**를 설계 및 개발
- 하드웨어 설계 조직, AI 연구 조직과 **긴밀한 협업** 필요
- 단순한 코딩이 아닌 **시스템 레벨의 아키텍처 설계**가 핵심

### 2️⃣ 수행업무 (Job Details) - 4가지 주요 업무

| 번호 | 업무 내용 | 핵심 역량 |
|------|----------|-----------|
| ① | 실시간 제어 프레임워크 설계·개발 | 시스템 아키텍처 설계 |
| ② | 모터·센서 제어용 디바이스 드라이버 개발 | 하드웨어-소프트웨어 연동 |
| ③ | Linux 기반 RTOS 환경 구성 및 최적화 | 임베디드 시스템 최적화 |
| ④ | AI 기능과 제어 시스템 간 연동 아키텍처 | 미들웨어 개발 |

### 3️⃣ 자격요건 (Requirements) - 필수 요건

- ✅ **전공**: 컴퓨터, 전기·전자, 기계, 로봇공학 등
- ✅ **운영체제 개념 이해** (필수)
- ✅ **소프트웨어 구조적 설계/구현 역량**
- ✅ **다학제 협업 소통 능력** (로봇공학은 융합 분야이므로 매우 중요)

### 4️⃣ 우대사항 (Preferences) - 가산점

| 기술 영역 | 세부 사항 |
|----------|----------|
| 프로그래밍 | **C/C++** 기반 시스템 프로그래밍 |
| 운영체제 | Linux 임베

In [ ]:
def compare_images(image_paths, question):
    content = []

    for i, path in enumerate(image_paths,start=1):
        img_b64 = encode_image(path)
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}
        })
        content.append({"type": "text", "text": f"[이미지 {i}]"})

    # 질문 추가
    content.append({"type":"text", "text":question})
    message = HumanMessage(content=content)
    response = vision_llm.invoke([message])


    return response.content

result = compare_images(['./image/fridge.jpg', './image/table.jpg'], '두 제품의 차이점을 비교 분석 해주세요.')
print(result)

In [15]:
result = compare_images(
    ["./image/chart1.png", "./image/chart2.png"],
    "두 차트를 비교하여 주요 변화 수치를 설명해 주세요",
)
print(result)


# 두 차트 비교 분석: 화장품 수출 동향

## 📊 차트 1: 최근 6개월 전체 화장품 수출액 추이

| 기간 | 수출액(백만$) | YoY 증감률 |
|------|--------------|------------|
| 25년 12월 | 883.7 | +10.2% |
| 26년 1월 | 841.6 | **+33.7%** ← 최고 |
| 26년 2월 | 752.1 | +1.7% |
| 26년 3월 | 960.4 | +23.0% |
| 26년 4월 | **1,096.3** | +21.7% |
| 26년 5월(1~20일) | 671.1 | **−16.0%** ← 최저 |

## 📊 차트 2: 5월 1~20일 주요 국가별 일평균 수출액

| 국가 | 일평균 수출액(백만$) | YoY 증감률 |
|------|---------------------|------------|
| 중화권(중국) | 13.13 | **+76.4%** |
| 미국 | 11.66 | +40.3% |
| 유럽 | 11.14 | +61.3% |
| 동남아 | 4.91 | +22.0% |
| 일본 | 약 3.0 | **−14.1%** |

---

## 🔍 주요 변화 수치 및 시사점

### 1. **전체 수출의 급격한 반전**
- 1월(+33.7%) → 4월(+21.7%)까지 **꾸준한 고성장**을 유지했으나
- 5월 1~20일 기준 **−16.0%**로 전환되며 **약 50%p 폭락**했습니다.

### 2. **국가별 양극화 현상**
5월 전체적으로 마이너스임에도 불구하고, 주요 5개국 중 **4개국은 여전히 +22~76%대 고성장**을 기록했습니다.
- ✅ **중화권: +76.4%** (압도적 1위)
- ✅ **유럽: +61.3%**
- ✅ **미국: +40.3%**
- ❌ **일본: −14.1%** (전체 하락의 주요 원인 추정)

### 3. **핵심 인사이트**
- 전체 −16% 하락은 **주요 거대 시장(중·미·유럽)의 성장세가 무색할 정도로** 일본을 포함한 중소 시장 또는 기타 